# URB-05 · Notebook piloto: simulación de escenarios preventivos urbanos en un gemelo digital

**Proyecto FCI / RISKTWIN — Gemelo digital multimodal urbano-industrial**
Universidad de Guayaquil · Facultad de Ingeniería Industrial · Carrera de Sistemas de la Información

**Autores:** Bryan Alexis Estupiñan Marcillo · Angel David Leon Muirragui  |  **Versión del módulo:** 0.1.0

---

## ¿Qué es este notebook?

Es una **prueba de concepto reproducible** del módulo `simulation.py` descrito en el anteproyecto *"Simulación de escenarios preventivos urbanos en un gemelo digital multimodal"*. Responde a la pregunta de investigación: **¿cómo evaluar escenarios de intervención preventiva mediante simulación en un gemelo digital urbano?**

Idea central: antes de cambiar una ruta, restringir un carril o crear una zona peatonal en la calle, se **simula** el cambio sobre el índice de criticidad urbana y se compara contra un **escenario base**. La simulación usa:

- **Simulación de Eventos Discretos (DES)** como metodología principal: el estado de cada tramo vial cambia solo en instantes discretos (cambio de franja horaria, inicio de una intervención, inicio y fin de un incidente), sin simular vehículos ni personas individuales.
- **Monte Carlo** para introducir variabilidad (demanda de tráfico, incidentes, incertidumbre sobre cuánto funcionará cada intervención).
- **Validación estadística formal:** prueba de Kolmogorov-Smirnov (K-S) del escenario base, ANOVA / Kruskal-Wallis para comparar escenarios y análisis de sensibilidad.

### Qué se entrega en cada etapa

| Objetivo específico del anteproyecto | Dónde se aborda |
|---|---|
| **OE1.** Definir escenarios, reglas, parámetros y criterios a partir de los insumos de FCI-03 | §3, §4 y §7 |
| **OE2.** Módulo de simulación basado en reglas, que ejecuta y compara escenarios contra el base | §3 (`simulation.py`), §5, §6, §8 y §9 |
| **OE3.** Evaluar reducción estimada de criticidad y sensibilidad de parámetros | §9, §10 y §11 |
| **OE4.** Visualización y exportación para el dashboard / API de FCI | §12, §13 y §14 |

| Historia de usuario | Sección |
|---|---|
| **HU-01** Cargar el índice de criticidad base y los parámetros (URB-01, URB-02, URB-03) | §4 |
| **HU-02** Ejecutar escenarios preventivos con reglas lógicas | §7 y §8 |
| **HU-03** Aplicar variabilidad Monte Carlo y analizar sensibilidad | §9 y §11 |
| **HU-04** Comparativa gráfica base vs. intervenido (tablero interactivo) | §12 |
| **HU-05** Exportar resultados en formatos estandarizados (API) | §13 y §14 |

### Contenido

0. Preparación del entorno
1. Parámetros generales
2. Marco metodológico: ciclo de un estudio de simulación y DES frente a ABM
3. Modelo conceptual y motor de simulación (`simulation.py`)
4. Insumos de FCI y carga del escenario base (HU-01)
5. Verificación del modelo (pruebas modulares)
6. Validación del escenario base (K-S y calibración)
7. Catálogo de escenarios (HU-02)
8. Una corrida DES paso a paso
9. Monte Carlo y comparación de escenarios (HU-03)
10. Diseño de experimentos: ANOVA / Kruskal-Wallis
11. Análisis de sensibilidad y robustez del ranking
12. Tablero y mapas (HU-04)
13. Exportación e integración con el dashboard / API (HU-05)
14. Pruebas de integración, regresión y estrés
15. Conclusiones, limitaciones y próximos pasos

> **Aviso importante.** El anteproyecto establece que el módulo usa **insumos precalculados por FCI-01, FCI-02 y FCI-03**. Como esos insumos aún no están disponibles, este piloto genera un **insumo sintético** con el contrato de datos esperado (o lo carga de un CSV / del dataset de URB-01 si existe). Los resultados son **estimaciones bajo supuestos documentados, no predicciones del comportamiento real de la ciudad**, tal como lo declara el alcance del proyecto.

**Tiempo estimado de ejecución:** 1 a 3 minutos (*Entorno de ejecución → Ejecutar todo*).

## 0. Preparación del entorno

Colab ya incluye NumPy, pandas, SciPy, Matplotlib y GeoPandas. Se instalan solo **Folium** (mapas) y **Plotly** (gráficos interactivos) por si faltan.

In [ ]:
!pip install -q folium plotly

In [ ]:
import os, json, hashlib, shutil, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 170)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
print("NumPy", np.__version__, "| pandas", pd.__version__)

## 1. Parámetros generales

Se centralizan aquí los parámetros que controlan el piloto. Cambiar `N_REP` modifica la precisión del análisis Monte Carlo (más réplicas = intervalos más estrechos, más tiempo).

**Origen del insumo (`ORIGEN_INSUMO`):**

| Valor | Qué hace |
|---|---|
| `"sintetico"` | Genera una red urbana piloto sintética con el contrato de datos de FCI (por defecto). |
| `"csv"` | Lee `insumo_fci_segmentos.csv` entregado por FCI-01/02/03 (mismas columnas que el sintético). |
| `"urb01"` | Construye el insumo a partir del GeoPackage del módulo URB-01 (`urb01_dataset_v0_1.gpkg`), si se ejecutó antes. |

In [ ]:
SEMILLA = 42
N_REP = 200            # réplicas Monte Carlo por escenario (análisis principal)
N_REP_SENS = 30        # réplicas por punto del análisis de sensibilidad
K_OBJETIVO = 12        # tramos intervenidos en cada escenario

ORIGEN_INSUMO = "sintetico"                                   # "sintetico" | "csv" | "urb01"
RUTA_CSV = "insumo_fci_segmentos.csv"
RUTA_URB01 = "urb01_salida/urb01_dataset_v0_1.gpkg"

SALIDA = Path("urb05_salida")
SALIDA.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(SEMILLA)

## 2. Marco metodológico

### 2.1 Ciclo de vida de un estudio de simulación

El anteproyecto adopta el ciclo de un estudio de simulación de **Banks et al.** como marco de rigor. Este notebook recorre sus etapas en orden; la tabla muestra dónde está cada una:

| Etapa del estudio de simulación | Qué se hace aquí | Sección |
|---|---|---|
| Formulación del problema y objetivos | Ya definidos en el anteproyecto (sección 1 y 4) | — |
| Conceptualización del modelo | Variables de estado, eventos, reglas de intervención, supuestos | §3 |
| Recolección de datos | Insumos de FCI y contrato de datos | §4 |
| Traducción del modelo a código | Archivo `simulation.py` | §3 |
| **Verificación** (¿el código hace lo que se diseñó?) | Pruebas modulares | §5 |
| **Validación** (¿el modelo se parece a la realidad?) | K-S frente a datos históricos, correlación con el índice de FCI-03 | §6 |
| Diseño experimental | Escenarios, réplicas, números aleatorios comunes | §7 y §9 |
| Corridas y análisis | Monte Carlo, ANOVA / Kruskal-Wallis, sensibilidad | §9 a §11 |
| Documentación y reporte | Tablero, exportaciones | §12 y §13 |
| Implementación | Contrato JSON para el dashboard / API | §13 y §14 |

### 2.2 Por qué DES y no un modelo basado en agentes (ABM)

| | **DES (elegido)** | ABM |
|---|---|---|
| Unidad de modelado | Tramo vial con variables de estado | Cada vehículo / peatón como agente |
| Cambios de estado | En instantes discretos (eventos) | En cada paso de tiempo, por agente |
| Costo computacional | Bajo: un día simulado toma milisegundos | Alto: requiere arquitectura distribuida |
| Pregunta que responde bien | *¿Cuánto cambia la criticidad de la zona si intervengo estos tramos?* | *¿Cómo decide cada conductor?* |

Como el proyecto evalúa escenarios a nivel de zona y con restricciones de presupuesto y tiempo (4 meses), DES es la técnica adecuada.

### 2.3 Diseño experimental: números aleatorios comunes

Cada réplica usa **la misma semilla para todos los escenarios**: misma demanda y mismos incidentes en el escenario base y en los intervenidos. Así, la diferencia entre escenarios se debe a la intervención y no al azar. Esta técnica (*common random numbers*) hace que las comparaciones sean **pareadas** y por eso los intervalos de confianza salen estrechos; es esperable y se discute en §9.

## 3. Modelo conceptual y motor de simulación (`simulation.py`)

### 3.1 Modelo conceptual

**Índice de criticidad (0 a 100) por tramo y hora**, a partir de los tres indicadores que el anteproyecto menciona (flujo vehicular, exposición peatonal, infraestructura) más su interacción:

> `criticidad = 100 × ( 0.30·F + 0.25·P + 0.20·I + 0.25·F·P )`
>
> **F** = saturación vehicular (flujo / capacidad, acotada) · **P** = exposición peatonal · **I** = deterioro de la infraestructura · **F·P** = conflicto vehículo-peatón

**Variables de estado** de cada tramo: flujo vehicular, capacidad, exposición peatonal, deterioro de infraestructura.

**Eventos discretos** que cambian el estado:

| Evento | Qué ocurre |
|---|---|
| `CAMBIO_FRANJA` | Cada hora cambian los multiplicadores de demanda vehicular y peatonal (perfil de un día laborable). |
| `INTERV_ON` / `INTERV_OFF` | Entra o sale de vigencia una intervención preventiva (con ventana horaria). |
| `INCIDENTE` | Llega un incidente vial aleatorio (proceso de Poisson) que reduce la capacidad del tramo, elegido con mayor probabilidad donde la criticidad ya es alta. Parte de su flujo se derrama a los vecinos. |
| `FIN_INCIDENTE` | El tramo recupera su capacidad. |
| `MUESTREO` | Se registra la criticidad de todos los tramos. |

**Intervenciones preventivas simples** (las cuatro que menciona el anteproyecto):

| Tipo | Regla |
|---|---|
| `desvio_ruta` | Una fracción del flujo del tramo se desvía hacia tramos vecinos. |
| `restriccion_carril` | Baja la capacidad; la demanda cae en proporción (con elasticidad); el flujo retirado va a los vecinos; hay un pequeño beneficio peatonal. |
| `zona_peatonal` | Se restringe el tráfico vehicular (el flujo va a los vecinos) y baja la exposición peatonal al riesgo. |
| `senalizacion` | Mejora la infraestructura y da un pequeño beneficio peatonal. |

> **Principio clave: el flujo desviado no desaparece.** Se redistribuye entre los vecinos no intervenidos. Esto permite responder una pregunta central del anteproyecto: si una política *reduce el riesgo o solo traslada el conflicto a otras zonas*. Por eso el módulo mide tres cosas: la criticidad en los **tramos intervenidos**, en su **zona de influencia** (intervenidos + vecinos directos) y en **toda la ciudad**.

### 3.2 Supuestos documentados

Todo parámetro del modelo es un **supuesto** del equipo hasta que se calibre con datos reales. La celda siguiente los lista y los guarda en `supuestos_parametros.csv`, que corresponde a la fuente *"Parámetros de intervención y supuestos documentados"* de la sección 8.3 del anteproyecto.

### 3.3 Traducción a código

La celda siguiente escribe el archivo `simulation.py` (el módulo que nombra el alcance del proyecto). Está organizada en ocho bloques comentados: configuración, criticidad, red, intervenciones, motor DES, Monte Carlo, sensibilidad y contrato JSON. Se escribe desde el notebook para que **todo el piloto sea autocontenido**, y luego se importa como cualquier módulo.

In [ ]:
%%writefile simulation.py
"""
simulation.py - Motor de simulación de escenarios preventivos urbanos (URB-05 / RISKTWIN)

Metodología: Simulación de Eventos Discretos (DES) con cola de eventos, más Monte Carlo
para incertidumbre y análisis de sensibilidad. No simula agentes individuales (ABM): el
estado del sistema son variables por tramo vial que cambian en instantes discretos.

Contenido
  1. Constantes y configuración
  2. Modelo de criticidad
  3. Red urbana (insumos de FCI-01/02/03)
  4. Intervenciones y escenarios
  5. Motor DES
  6. Monte Carlo y comparación
  7. Análisis de sensibilidad
  8. Contrato de intercambio (JSON) para el dashboard / API
"""
from __future__ import annotations

import copy
import heapq
import itertools
import json
from dataclasses import dataclass, field, asdict
from datetime import datetime
from typing import Dict, List, Optional

import numpy as np
import pandas as pd

VERSION = "0.1.0"
ESQUEMA_RESULTADOS = "risktwin.urb05.resultados/1.0"

# ----------------------------------------------------------------------------
# 1. Constantes y configuración
# ----------------------------------------------------------------------------
# Capacidad nominal por jerarquía vial (veh/h). SUPUESTO documentado: calibrar con ATM.
CAPACIDAD_VEH_H = {"motorway": 3600, "trunk": 2400, "primary": 1800,
                   "secondary": 1200, "tertiary": 800, "residential": 400}
CAPACIDAD_DEFECTO = 600

# Perfiles horarios (multiplicadores con media 1). SUPUESTO: día laborable típico.
_PV = np.array([.25, .18, .15, .15, .25, .60, 1.2, 1.8, 1.7, 1.2, 1.0, 1.0,
                1.1, 1.1, 1.0, 1.1, 1.4, 1.8, 1.7, 1.2, .8, .6, .45, .3])
_PP = np.array([.10, .05, .05, .05, .10, .30, .8, 1.2, 1.1, 1.0, 1.0, 1.3,
                1.5, 1.2, 1.0, 1.1, 1.3, 1.5, 1.2, .9, .6, .4, .25, .15])
PERFIL_VEH = _PV / _PV.mean()
PERFIL_PEAT = _PP / _PP.mean()

PESOS_DEFECTO = {"flujo": 0.30, "peatonal": 0.25, "infra": 0.20, "conflicto": 0.25}

# Parámetros por defecto de cada tipo de intervención (todos son fracciones 0-1)
PARAMS_DEFECTO = {
    "desvio_ruta":        {"fraccion_desvio": 0.50},
    "restriccion_carril": {"reduccion_capacidad": 0.30, "elasticidad_demanda": 1.20,
                           "beneficio_peatonal": 0.15},
    "zona_peatonal":      {"restriccion_vehicular": 0.80, "reduccion_exposicion": 0.60},
    "senalizacion":       {"mejora_infra": 0.35, "beneficio_peatonal": 0.05},
}


@dataclass
class ConfigSim:
    horizonte_h: int = 24                 # un día típico
    sigma_demanda: float = 0.25           # variabilidad lognormal de la demanda por tramo y hora
    tasa_incidentes_dia: float = 6.0      # incidentes viales aleatorios en toda la zona por día
    duracion_incidente_h: float = 0.75    # duración media (exponencial)
    factor_cap_incidente: float = 0.50    # capacidad residual del tramo con incidente
    derrame_incidente: float = 0.15       # fracción del flujo que se desvía a los vecinos
    umbral_critico: float = 60.0          # criticidad a partir de la cual un tramo-hora es "crítico"
    incertidumbre_parametros: float = 0.0 # +/- fracción relativa sobre los parámetros de intervención
    pesos: Dict[str, float] = field(default_factory=lambda: dict(PESOS_DEFECTO))

    def __post_init__(self):
        if abs(sum(self.pesos.values()) - 1.0) > 1e-9:
            raise ValueError("Los pesos de criticidad deben sumar 1.")


# ----------------------------------------------------------------------------
# 2. Modelo de criticidad
# ----------------------------------------------------------------------------
def criticidad(flujo, cap, peat, infra, pesos):
    # Índice 0-100: saturación vehicular, exposición peatonal, deterioro y su conflicto.
    sat = np.minimum(flujo / np.maximum(cap, 1.0), 1.5) / 1.5
    p = np.clip(peat, 0.0, 1.0)
    i = np.clip(infra, 0.0, 1.0)
    return 100.0 * (pesos["flujo"] * sat + pesos["peatonal"] * p
                    + pesos["infra"] * i + pesos["conflicto"] * sat * p)


# ----------------------------------------------------------------------------
# 3. Red urbana (insumos)
# ----------------------------------------------------------------------------
COLUMNAS_INSUMO = ["id_segmento", "id_zona", "tipo_via", "longitud_m", "flujo_veh_h",
                   "exposicion_peatonal", "deterioro_infra", "nodo_origen", "nodo_destino",
                   "lat", "lon"]


def validar_insumo(df: pd.DataFrame) -> List[str]:
    # Devuelve la lista de problemas del insumo (vacía = válido). Contrato de HU-01.
    faltan = [c for c in COLUMNAS_INSUMO if c not in df.columns]
    if faltan:
        return [f"Faltan columnas obligatorias: {faltan}"]
    p = []
    if df["id_segmento"].duplicated().any():
        p.append("id_segmento duplicado")
    nulos = df[COLUMNAS_INSUMO].isna().sum()
    p += [f"Nulos en '{c}': {int(n)}" for c, n in nulos.items() if n > 0]
    for c in ("exposicion_peatonal", "deterioro_infra"):
        if ((df[c] < 0) | (df[c] > 1)).any():
            p.append(f"'{c}' fuera de [0, 1]")
    if (df["flujo_veh_h"] < 0).any():
        p.append("flujo_veh_h negativo")
    if (df["longitud_m"] <= 0).any():
        p.append("longitud_m no positiva")
    return p


@dataclass
class Red:
    ids: np.ndarray
    zona: np.ndarray
    tipo_via: np.ndarray
    longitud: np.ndarray
    flujo: np.ndarray
    cap: np.ndarray
    peat: np.ndarray
    infra: np.ndarray
    lat: np.ndarray
    lon: np.ndarray
    vecinos: List[np.ndarray]
    indice_fci: Optional[np.ndarray] = None

    @property
    def n(self) -> int:
        return len(self.ids)

    def indices(self, ids) -> np.ndarray:
        pos = {k: i for i, k in enumerate(self.ids)}
        return np.array([pos[i] for i in ids], dtype=int)


def construir_red(df: pd.DataFrame) -> Red:
    problemas = validar_insumo(df)
    if problemas:
        raise ValueError("Insumo inválido: " + "; ".join(problemas))
    df = df.reset_index(drop=True)
    por_nodo: Dict = {}
    for i, (a, b) in enumerate(zip(df["nodo_origen"], df["nodo_destino"])):
        por_nodo.setdefault(a, []).append(i)
        por_nodo.setdefault(b, []).append(i)
    vecinos = []
    for i, (a, b) in enumerate(zip(df["nodo_origen"], df["nodo_destino"])):
        v = set(por_nodo[a]) | set(por_nodo[b])
        v.discard(i)
        vecinos.append(np.array(sorted(v), dtype=int))
    cap = df["tipo_via"].map(lambda t: CAPACIDAD_VEH_H.get(t, CAPACIDAD_DEFECTO)).to_numpy(float)
    return Red(ids=df["id_segmento"].to_numpy(), zona=df["id_zona"].to_numpy(),
               tipo_via=df["tipo_via"].to_numpy(), longitud=df["longitud_m"].to_numpy(float),
               flujo=df["flujo_veh_h"].to_numpy(float), cap=cap,
               peat=df["exposicion_peatonal"].to_numpy(float),
               infra=df["deterioro_infra"].to_numpy(float),
               lat=df["lat"].to_numpy(float), lon=df["lon"].to_numpy(float), vecinos=vecinos,
               indice_fci=(df["indice_criticidad_fci03"].to_numpy(float)
                           if "indice_criticidad_fci03" in df.columns else None))


def indice_estatico(red: Red, pesos=None) -> np.ndarray:
    # Criticidad del estado promedio (sin perfil horario ni ruido). Equivale al índice "base" de FCI-03.
    return criticidad(red.flujo, red.cap, red.peat, red.infra, pesos or PESOS_DEFECTO)


# ----------------------------------------------------------------------------
# 4. Intervenciones y escenarios
# ----------------------------------------------------------------------------
@dataclass
class Intervencion:
    tipo: str                      # desvio_ruta | restriccion_carril | zona_peatonal | senalizacion
    objetivo: List[str]            # ids de tramos intervenidos
    params: Dict[str, float] = field(default_factory=dict)
    inicio_h: float = 0.0
    fin_h: float = 24.0

    def __post_init__(self):
        if self.tipo not in PARAMS_DEFECTO:
            raise ValueError(f"Tipo de intervención desconocido: {self.tipo}")
        self.params = {**PARAMS_DEFECTO[self.tipo], **self.params}


@dataclass
class Escenario:
    nombre: str
    intervenciones: List[Intervencion] = field(default_factory=list)
    descripcion: str = ""


def seleccionar_objetivo(red: Red, k: int = 10, zona: Optional[str] = None, ids=None) -> List[str]:
    # Regla de selección: los k tramos más críticos (opcionalmente de una zona) o una lista explícita.
    if ids is not None:
        return list(ids)
    c = indice_estatico(red)
    idx = np.arange(red.n) if zona is None else np.where(red.zona == zona)[0]
    idx = idx[np.argsort(-c[idx])][:k]
    return list(red.ids[idx])


def _redistribuir(flujo, obj_idx, quitado, red):
    # El flujo retirado de los tramos objetivo NO desaparece: pasa a los vecinos no intervenidos,
    # proporcional a su capacidad. Así se mide si el conflicto solo se traslada a otra zona.
    es_obj = np.zeros(red.n, bool)
    es_obj[obj_idx] = True
    for k, q in zip(obj_idx, quitado):
        vec = red.vecinos[k]
        vec = vec[~es_obj[vec]]
        if len(vec) == 0:
            flujo[k] += q            # sin salida posible: el flujo permanece
        else:
            w = red.cap[vec]
            flujo[vec] += q * w / w.sum()


def _aplicar(estado: dict, iv: Intervencion, red: Red):
    idx = red.indices(iv.objetivo)
    p = iv.params
    f = estado["flujo"]
    if iv.tipo == "desvio_ruta":
        quitado = f[idx] * p["fraccion_desvio"]
        f[idx] -= quitado
        _redistribuir(f, idx, quitado, red)
    elif iv.tipo == "restriccion_carril":
        r = p["reduccion_capacidad"]
        estado["cap"][idx] *= (1 - r)
        quitado = f[idx] * min(0.95, p["elasticidad_demanda"] * r)
        f[idx] -= quitado
        _redistribuir(f, idx, quitado, red)
        estado["peat"][idx] *= (1 - p["beneficio_peatonal"])
    elif iv.tipo == "zona_peatonal":
        quitado = f[idx] * p["restriccion_vehicular"]
        f[idx] -= quitado
        _redistribuir(f, idx, quitado, red)
        estado["peat"][idx] *= (1 - p["reduccion_exposicion"])
    elif iv.tipo == "senalizacion":
        estado["infra"][idx] *= (1 - p["mejora_infra"])
        estado["peat"][idx] *= (1 - p["beneficio_peatonal"])


def _perturbar(ivs, incert, rng_p):
    # Incertidumbre paramétrica (Monte Carlo). El uniforme se consume siempre, para que el flujo de
    # números aleatorios no dependa de 'incert' y los escenarios queden sincronizados.
    out = []
    for iv in ivs:
        iv2 = copy.deepcopy(iv)
        for k in sorted(iv2.params):
            u = rng_p.uniform(-1, 1)
            if incert > 0:
                iv2.params[k] = float(np.clip(iv2.params[k] * (1 + incert * u), 0.0, 1.5 if k == "elasticidad_demanda" else 0.95))
        out.append(iv2)
    return out


# ----------------------------------------------------------------------------
# 5. Motor DES
# ----------------------------------------------------------------------------
@dataclass
class Resultado:
    escenario: str
    semilla: int
    crit_t: np.ndarray      # (horas, tramos) criticidad en cada muestreo horario
    flujo_t: np.ndarray     # (horas, tramos) flujo efectivo en veh/h
    n_incidentes: int
    log: list

    def metricas(self, cfg: ConfigSim) -> dict:
        c = self.crit_t
        return {"crit_media": float(c.mean()),
                "crit_p95": float(np.percentile(c, 95)),
                "frac_critica": float((c > cfg.umbral_critico).mean()),
                "crit_max": float(c.max()),
                "n_incidentes": self.n_incidentes}


def simular(red: Red, escenario: Escenario, semilla: int = 0, cfg: Optional[ConfigSim] = None,
            registrar: bool = False) -> Resultado:
    # Tres flujos de números aleatorios independientes -> misma semilla = mismas condiciones
    # externas (demanda, incidentes) en todos los escenarios (números aleatorios comunes).
    cfg = cfg or ConfigSim()
    H, n = cfg.horizonte_h, red.n
    rng_d = np.random.default_rng([semilla, 1])
    rng_i = np.random.default_rng([semilla, 2])
    rng_p = np.random.default_rng([semilla, 3])
    s = cfg.sigma_demanda
    ruido = rng_d.lognormal(-0.5 * s * s, s, size=(H, n))          # media 1
    ivs = _perturbar(escenario.intervenciones, cfg.incertidumbre_parametros, rng_p)

    # --- estado del sistema ---
    base = {"flujo": red.flujo.copy(), "cap": red.cap.copy(),
            "peat": red.peat.copy(), "infra": red.infra.copy()}
    est = {k: v.copy() for k, v in base.items()}
    activas: List[Intervencion] = []
    cap_factor = np.ones(n)
    extra = np.zeros(n)
    hora = [0]
    crit_t = np.zeros((H, n))
    flujo_t = np.zeros((H, n))
    log: list = []

    def recalcular():
        nonlocal est
        est = {k: v.copy() for k, v in base.items()}
        for iv in activas:
            _aplicar(est, iv, red)

    def calcular():
        h = hora[0]
        flujo = est["flujo"] * PERFIL_VEH[h % 24] * ruido[h] + extra
        cap = est["cap"] * cap_factor
        peat = est["peat"] * PERFIL_PEAT[h % 24]
        return flujo, criticidad(flujo, cap, peat, est["infra"], cfg.pesos)

    # --- cola de eventos: (tiempo, prioridad, secuencia, tipo, dato) ---
    cola: list = []
    seq = itertools.count()

    def programar(t, prio, tipo, dato=None):
        heapq.heappush(cola, (t, prio, next(seq), tipo, dato))

    for h in range(H):
        programar(h, 0, "CAMBIO_FRANJA", h)
        programar(h + 0.5, 9, "MUESTREO", h)
    for iv in ivs:
        programar(iv.inicio_h, 1, "INTERV_ON", iv)
        if iv.fin_h < H:
            programar(iv.fin_h, 1, "INTERV_OFF", iv)
    t, n_inc = 0.0, 0
    lam = cfg.tasa_incidentes_dia / 24.0
    while lam > 0:
        t += rng_i.exponential(1 / lam)
        if t >= H:
            break
        programar(t, 5, "INCIDENTE", (rng_i.random(), rng_i.exponential(cfg.duracion_incidente_h)))

    # --- bucle principal: se procesa el evento más próximo hasta vaciar la cola ---
    while cola:
        t, _, _, tipo, dato = heapq.heappop(cola)
        if tipo == "CAMBIO_FRANJA":
            hora[0] = dato
        elif tipo == "INTERV_ON":
            activas.append(dato)
            recalcular()
        elif tipo == "INTERV_OFF":
            activas.remove(dato)
            recalcular()
        elif tipo == "INCIDENTE" and t < H:
            u, dur = dato
            flujo, c = calcular()
            cum = np.cumsum(c + 1e-9)
            k = int(min(np.searchsorted(cum, u * cum[-1]), n - 1))
            vec = red.vecinos[k]
            aporte = cfg.derrame_incidente * flujo[k] / max(1, len(vec))
            cap_factor[k] *= cfg.factor_cap_incidente
            extra[vec] += aporte
            n_inc += 1
            programar(t + dur, 4, "FIN_INCIDENTE", (k, vec, aporte))
            dato = {"tramo": str(red.ids[k]), "dur_h": round(dur, 2)}
        elif tipo == "FIN_INCIDENTE":
            k, vec, aporte = dato
            cap_factor[k] /= cfg.factor_cap_incidente
            extra[vec] = np.maximum(extra[vec] - aporte, 0.0)
            dato = {"tramo": str(red.ids[k])}
        elif tipo == "MUESTREO":
            flujo, c = calcular()
            flujo_t[dato], crit_t[dato] = flujo, c
        if registrar and tipo not in ("MUESTREO", "CAMBIO_FRANJA"):
            log.append({"t_h": round(t, 2), "evento": tipo,
                        "detalle": (dato if isinstance(dato, dict) else
                                    getattr(dato, "tipo", None))})
    return Resultado(escenario.nombre, semilla, crit_t, flujo_t, n_inc, log)


# ----------------------------------------------------------------------------
# 6. Monte Carlo y comparación
# ----------------------------------------------------------------------------
def zonas_escenario(red: Red, esc: Escenario):
    # (tramos intervenidos, zona de influencia = intervenidos + vecinos directos)
    ids = sorted({i for iv in esc.intervenciones for i in iv.objetivo})
    if not ids:
        v = np.array([], dtype=int)
        return v, v
    obj = red.indices(ids)
    infl = np.unique(np.concatenate([obj] + [red.vecinos[k] for k in obj]))
    return obj, infl


def _red_pct(b, e):
    return 100.0 * (b - e) / b if b > 0 else 0.0


@dataclass
class ResultadosMC:
    resumen: pd.DataFrame            # una fila por (escenario, réplica)
    crit_seg: Dict[str, np.ndarray]  # criticidad media por tramo (promedio de réplicas)
    crit_hora: Dict[str, np.ndarray] # criticidad media por hora (promedio de tramos y réplicas)
    objetivos: Dict[str, List[str]]

    def tabla(self) -> pd.DataFrame:
        g = self.resumen.groupby("escenario", sort=False)
        n = g.size()
        t = pd.DataFrame({
            "n_rep": n,
            "crit_media": g["crit_media"].mean(),
            "reduccion_obj_pct": g["reduccion_obj_pct"].mean(),
            "reduccion_infl_pct": g["reduccion_infl_pct"].mean(),
            "ic95_inf": g["reduccion_infl_pct"].mean() - 1.96 * g["reduccion_infl_pct"].std(ddof=1) / np.sqrt(n),
            "ic95_sup": g["reduccion_infl_pct"].mean() + 1.96 * g["reduccion_infl_pct"].std(ddof=1) / np.sqrt(n),
            "reduccion_ciudad_pct": g["reduccion_pct"].mean(),
            "prob_mejora": g["reduccion_infl_pct"].apply(lambda s: float((s > 0).mean())),
            "desplazamiento_max": g["desplazamiento_max"].mean(),
            "segmentos_empeorados": g["segmentos_empeorados"].mean(),
        })
        return t.sort_values("reduccion_infl_pct", ascending=False)


def monte_carlo(red: Red, escenarios: List[Escenario], n_rep: int = 100,
                cfg: Optional[ConfigSim] = None, semilla0: int = 1000) -> ResultadosMC:
    cfg = cfg or ConfigSim()
    base_sc = Escenario("Base", descripcion="Sin intervención")
    escs = [base_sc] + [e for e in escenarios if e.nombre != "Base"]
    objetivos = {e.nombre: sorted({i for iv in e.intervenciones for i in iv.objetivo}) for e in escs}
    es_obj = {e.nombre: np.isin(red.ids, objetivos[e.nombre]) for e in escs}
    zonas = {e.nombre: zonas_escenario(red, e) for e in escs}
    acum_seg = {e.nombre: np.zeros(red.n) for e in escs}
    acum_hora = {e.nombre: np.zeros(cfg.horizonte_h) for e in escs}
    filas = []
    for r in range(n_rep):
        s = semilla0 + r
        rb = simular(red, base_sc, s, cfg)
        mb, seg_b = rb.metricas(cfg), rb.crit_t.mean(0)
        for e in escs:
            res = rb if e.nombre == "Base" else simular(red, e, s, cfg)
            m = res.metricas(cfg)
            seg = res.crit_t.mean(0)
            d = seg - seg_b
            fuera = ~es_obj[e.nombre]
            obj, infl = zonas[e.nombre]
            r_obj = _red_pct(seg_b[obj].mean(), seg[obj].mean()) if len(obj) else 0.0
            r_inf = _red_pct(seg_b[infl].mean(), seg[infl].mean()) if len(infl) else 0.0
            filas.append({"escenario": e.nombre, "rep": r, "semilla": s, **m,
                          "reduccion_obj_pct": r_obj, "reduccion_infl_pct": r_inf,
                          "reduccion_abs": mb["crit_media"] - m["crit_media"],
                          "reduccion_pct": 100 * (mb["crit_media"] - m["crit_media"]) / mb["crit_media"],
                          "desplazamiento_max": float(d[fuera].max()) if fuera.any() else 0.0,
                          "segmentos_empeorados": int((d > 1.0).sum())})
            acum_seg[e.nombre] += seg
            acum_hora[e.nombre] += res.crit_t.mean(1)
    return ResultadosMC(pd.DataFrame(filas),
                        {k: v / n_rep for k, v in acum_seg.items()},
                        {k: v / n_rep for k, v in acum_hora.items()}, objetivos)


# ----------------------------------------------------------------------------
# 7. Análisis de sensibilidad
# ----------------------------------------------------------------------------
def _con_parametro(esc: Escenario, cfg: ConfigSim, clave: str, valor: float):
    # Claves: "cfg.sigma_demanda"  |  "iv0.fraccion_desvio" (intervención 0, parámetro ...)
    esc, cfg = copy.deepcopy(esc), copy.deepcopy(cfg)
    if clave.startswith("cfg."):
        setattr(cfg, clave[4:], valor)
    else:
        i, p = clave[2:].split(".", 1)
        esc.intervenciones[int(i)].params[p] = valor
    return esc, cfg


def _valor_central(esc: Escenario, cfg: ConfigSim, clave: str) -> float:
    if clave.startswith("cfg."):
        return float(getattr(cfg, clave[4:]))
    i, p = clave[2:].split(".", 1)
    return float(esc.intervenciones[int(i)].params[p])


def reduccion_media_pct(red, esc, cfg, n_rep=30, semilla0=5000) -> float:
    # Métrica de sensibilidad: reducción media de criticidad en la zona de influencia (%).
    base = Escenario("Base")
    _, infl = zonas_escenario(red, esc)
    v = []
    for r in range(n_rep):
        s = semilla0 + r
        cb = simular(red, base, s, cfg).crit_t.mean(0)[infl].mean()
        ce = simular(red, esc, s, cfg).crit_t.mean(0)[infl].mean()
        v.append(_red_pct(cb, ce))
    return float(np.mean(v))


def sensibilidad_oat(red, esc, cfg, rangos: Dict[str, tuple], n_rep=30, semilla0=5000) -> pd.DataFrame:
    # Un parámetro a la vez (tornado): se mueve cada parámetro a su mínimo y máximo.
    central = reduccion_media_pct(red, esc, cfg, n_rep, semilla0)
    filas = []
    for clave, (lo, hi) in rangos.items():
        r = []
        for v in (lo, hi):
            e2, c2 = _con_parametro(esc, cfg, clave, v)
            r.append(reduccion_media_pct(red, e2, c2, n_rep, semilla0))
        filas.append({"parametro": clave, "valor_central": _valor_central(esc, cfg, clave),
                      "min": lo, "max": hi, "reduccion_en_min": r[0], "reduccion_en_max": r[1],
                      "reduccion_central": central, "rango_abs": abs(r[1] - r[0])})
    return pd.DataFrame(filas).sort_values("rango_abs", ascending=False).reset_index(drop=True)


def sensibilidad_global(red, esc, cfg, rangos: Dict[str, tuple], n_muestras=64, n_rep=8,
                        semilla=7) -> pd.DataFrame:
    # Todos los parámetros varían a la vez (hipercubo latino); se mide la correlación de rangos (Spearman).
    from scipy.stats import qmc, spearmanr
    claves = list(rangos)
    lo = np.array([rangos[k][0] for k in claves])
    hi = np.array([rangos[k][1] for k in claves])
    X = qmc.scale(qmc.LatinHypercube(d=len(claves), seed=semilla).random(n_muestras), lo, hi)
    y = []
    for fila in X:
        e2, c2 = esc, cfg
        for k, v in zip(claves, fila):
            e2, c2 = _con_parametro(e2, c2, k, float(v))
        y.append(reduccion_media_pct(red, e2, c2, n_rep, 9000))
    y = np.array(y)
    rho = [spearmanr(X[:, j], y)[0] for j in range(len(claves))]
    out = pd.DataFrame({"parametro": claves, "spearman": rho})
    out["abs"] = out["spearman"].abs()
    return out.sort_values("abs", ascending=False).reset_index(drop=True)


# ----------------------------------------------------------------------------
# 8. Contrato de intercambio (JSON) para el dashboard / API
# ----------------------------------------------------------------------------
def _resolver_objetivo(red: Red, obj) -> List[str]:
    return seleccionar_objetivo(red, **obj) if isinstance(obj, dict) else list(obj)


def escenario_desde_dict(red: Red, d: dict) -> Escenario:
    ivs = [Intervencion(tipo=i["tipo"], objetivo=_resolver_objetivo(red, i["objetivo"]),
                        params=i.get("params", {}), inicio_h=i.get("inicio_h", 0.0),
                        fin_h=i.get("fin_h", 24.0)) for i in d.get("intervenciones", [])]
    return Escenario(d["nombre"], ivs, d.get("descripcion", ""))


def resultados_a_dict(red: Red, mc: ResultadosMC, cfg: ConfigSim) -> dict:
    tabla = mc.tabla().reset_index()
    seg = []
    base = mc.crit_seg["Base"]
    for nombre, arr in mc.crit_seg.items():
        for i in range(red.n):
            seg.append({"id_segmento": str(red.ids[i]), "id_zona": str(red.zona[i]),
                        "escenario": nombre, "criticidad_media": round(float(arr[i]), 3),
                        "delta_vs_base": round(float(arr[i] - base[i]), 3)})
    return {"esquema": ESQUEMA_RESULTADOS, "version_modulo": VERSION,
            "generado": datetime.now().isoformat(timespec="seconds"),
            "configuracion": asdict(cfg),
            "escenarios": json.loads(tabla.round(4).to_json(orient="records")),
            "resultados_segmento": seg}


def ejecutar_desde_dict(red: Red, payload: dict) -> dict:
    # Punto de entrada para una API: recibe JSON de configuración y devuelve JSON de resultados.
    cfg = ConfigSim(**payload.get("cfg", {}))
    escs = [escenario_desde_dict(red, e) for e in payload["escenarios"]]
    mc = monte_carlo(red, escs, n_rep=payload.get("n_rep", 50), cfg=cfg,
                     semilla0=payload.get("semilla", 1000))
    return resultados_a_dict(red, mc, cfg)

In [ ]:
import importlib
import simulation as sim
importlib.reload(sim)
print("simulation.py cargado · versión", sim.VERSION)

# ---- Supuestos documentados (se exportan como tabla) ----
cfg_def = sim.ConfigSim()
filas = []
for t, c in sim.CAPACIDAD_VEH_H.items():
    filas.append(("Capacidad nominal", t, c, "veh/h", "Supuesto; calibrar con ATM / aforos"))
filas += [("Perfil horario vehicular", "24 multiplicadores (media 1)", "ver PERFIL_VEH", "—", "Supuesto: día laborable típico"),
          ("Perfil horario peatonal", "24 multiplicadores (media 1)", "ver PERFIL_PEAT", "—", "Supuesto: día laborable típico")]
for k, v in cfg_def.pesos.items():
    filas.append(("Peso de criticidad", k, v, "fracción", "Supuesto; idealmente tomar de FCI-03"))
for k in ("sigma_demanda", "tasa_incidentes_dia", "duracion_incidente_h", "factor_cap_incidente",
          "derrame_incidente", "umbral_critico", "incertidumbre_parametros"):
    filas.append(("Configuración", k, getattr(cfg_def, k), "", "Supuesto; sigma_demanda se calibra en §6"))
for t, d in sim.PARAMS_DEFECTO.items():
    for k, v in d.items():
        filas.append((f"Intervención: {t}", k, v, "fracción", "Supuesto de efectividad; sin evidencia de campo"))
supuestos = pd.DataFrame(filas, columns=["grupo", "parametro", "valor", "unidad", "origen / estado"])
supuestos.to_csv(SALIDA / "supuestos_parametros.csv", index=False, encoding="utf-8-sig")
print(f"{len(supuestos)} supuestos documentados")
display(supuestos.head(12))

## 4. Insumos de FCI y carga del escenario base (HU-01)

### 4.1 Contrato de datos

El módulo espera **una fila por tramo vial** con las siguientes columnas. Es el acuerdo de interfaz con FCI-01, FCI-02 y FCI-03 (cronograma: *"definición de interfaces de conexión y formatos de intercambio"*).

| Columna | Origen esperado | Descripción |
|---|---|---|
| `id_segmento`, `id_zona`, `tipo_via`, `longitud_m` | URB-01 / URB-02 | Identificación y jerarquía del tramo |
| `nodo_origen`, `nodo_destino` | URB-02 (grafo) | Permiten saber qué tramos son vecinos |
| `flujo_veh_h` | URB-01 (aforos) | Flujo vehicular medio por hora |
| `exposicion_peatonal` (0–1) | URB-01 / URB-03 | Exposición de peatones |
| `deterioro_infra` (0–1) | URB-01 / URB-03 | Estado de la infraestructura |
| `lat`, `lon` | URB-01 | Centro del tramo (opcionales: `lat_o`, `lon_o`, `lat_d`, `lon_d` para dibujarlo) |
| `indice_criticidad_fci03` | URB-03 | Índice de criticidad base (opcional, se usa para validar) |

`validar_insumo()` revisa columnas, nulos, rangos e identificadores **antes** de simular: un insumo inválido se rechaza con un mensaje claro (HU-01).

### 4.2 Tres formas de obtener el insumo

La celda siguiente define las tres: un generador sintético, la lectura de un CSV y un adaptador que construye el insumo desde el GeoPackage de URB-01 (aforos como flujo vehicular y puntos de interés como proxy de exposición peatonal; el deterioro de infraestructura, que URB-01 aún no tiene, se simula y queda declarado).

In [ ]:
def generar_insumo_sintetico(semilla=42, n=9, espaciado=120.0, centro=(-2.1950, -79.8860)):
    # Red en cuadrícula de n x n intersecciones; más densa y crítica hacia el centro.
    r = np.random.default_rng(semilla)
    lat0, lon0 = centro
    m_lat, m_lon = 111_320.0, 111_320.0 * np.cos(np.radians(lat0))
    xy = {(i, j): (i * espaciado + r.normal(0, 15), j * espaciado + r.normal(0, 15))
          for i in range(n) for j in range(n)}
    nid = {k: f"N{idx:03d}" for idx, k in enumerate(xy)}
    cx = cy = (n - 1) * espaciado / 2
    filas = []
    for (i, j), (x, y) in xy.items():
        for di, dj in [(1, 0), (0, 1)]:
            q = (i + di, j + dj)
            if q not in xy:
                continue
            x2, y2 = xy[q]
            mx, my = (x + x2) / 2, (y + y2) / 2
            d = np.hypot(mx - cx, my - cy) / (cx * 1.4)          # 0 en el centro, ~1 en el borde
            if d > 0.35:
                tipo = r.choice(["primary", "secondary", "tertiary", "residential"], p=[.15, .25, .25, .35])
            else:
                tipo = r.choice(["primary", "secondary", "tertiary"], p=[.4, .4, .2])
            cap = sim.CAPACIDAD_VEH_H[tipo]
            filas.append({
                "id_segmento": f"S{len(filas) + 1:04d}",
                "id_zona": ("Z1" if mx < cx else "Z2") + ("N" if my >= cy else "S"),
                "tipo_via": tipo, "longitud_m": round(float(np.hypot(x2 - x, y2 - y)), 1),
                "flujo_veh_h": round(float(np.clip(r.beta(4, 5) + 0.25 * (1 - d), 0.1, 0.85) * cap), 1),
                "exposicion_peatonal": round(float(np.clip(r.beta(2, 4) + 0.35 * (1 - d), 0.02, 0.95)), 3),
                "deterioro_infra": round(float(np.clip(r.beta(2, 5) + 0.2 * d, 0.02, 0.95)), 3),
                "nodo_origen": nid[(i, j)], "nodo_destino": nid[q],
                "lat": round(lat0 + (my - cy) / m_lat, 6), "lon": round(lon0 + (mx - cx) / m_lon, 6),
                "lat_o": round(lat0 + (y - cy) / m_lat, 6), "lon_o": round(lon0 + (x - cx) / m_lon, 6),
                "lat_d": round(lat0 + (y2 - cy) / m_lat, 6), "lon_d": round(lon0 + (x2 - cx) / m_lon, 6)})
    df = pd.DataFrame(filas)
    # Stand-in del índice base de FCI-03 (en el proyecto real lo entrega FCI-03)
    df["indice_criticidad_fci03"] = sim.indice_estatico(sim.construir_red(df)).round(2)
    return df

def generar_historico_sintetico(red, semilla=7, n_conteo=40, dias=7, sigma_real=0.27):
    # "Datos históricos" de aforo horario en tramos con conteo (en el proyecto real: ATM / FCI-01).
    r = np.random.default_rng(semilla)
    idx = r.choice(red.n, n_conteo, replace=False)
    bloques = []
    for d in range(dias):
        for h in range(24):
            ruido = r.lognormal(-0.5 * sigma_real ** 2, sigma_real, n_conteo)
            bloques.append(pd.DataFrame({"id_segmento": red.ids[idx], "dia": d, "hora": h,
                                         "flujo_veh_h": red.flujo[idx] * sim.PERFIL_VEH[h] * ruido}))
    return pd.concat(bloques, ignore_index=True)

def cargar_desde_urb01(ruta):
    # Adaptador URB-01 -> contrato de FCI. Devuelve (insumo, histórico de aforos).
    import geopandas as gpd
    seg = gpd.read_file(ruta, layer="segmento_vial")
    afo = gpd.read_file(ruta, layer="aforo")
    poi = gpd.read_file(ruta, layer="poi")
    seg_m, poi_m = seg.to_crs("EPSG:32717"), poi.to_crs("EPSG:32717")
    ctr = seg_m.geometry.interpolate(0.5, normalized=True).to_crs("EPSG:4326")
    cap = seg["tipo_via"].map(lambda t: sim.CAPACIDAD_VEH_H.get(t, sim.CAPACIDAD_DEFECTO))
    aforo_medio = afo.groupby("id_segmento")["vehiculos_hora"].mean()
    flujo = seg["id_segmento"].map(aforo_medio)
    flujo = flujo.fillna(flujo.groupby(seg["tipo_via"]).transform("median")).fillna(cap * 0.35)
    buf = seg_m[["id_segmento", "geometry"]].copy()
    buf["geometry"] = buf.geometry.buffer(150)
    n_poi = gpd.sjoin(poi_m[["geometry"]], buf, predicate="within").groupby("id_segmento").size()
    peat = (0.10 + 0.10 * seg["id_segmento"].map(n_poi).fillna(0).clip(upper=5)
            + np.where(seg["tipo_via"].isin(["tertiary", "residential"]), 0.10, 0.0)).clip(0.02, 0.95)
    r = np.random.default_rng(SEMILLA)
    ends = seg.geometry.apply(lambda g: (g.coords[0], g.coords[-1]))
    lat_c, lon_c = ctr.y.to_numpy(), ctr.x.to_numpy()
    zona = np.where(lon_c < np.median(lon_c), "Z1", "Z2") + np.where(lat_c >= np.median(lat_c), "N", "S")
    df = pd.DataFrame({
        "id_segmento": seg["id_segmento"], "id_zona": zona, "tipo_via": seg["tipo_via"],
        "longitud_m": seg["longitud_m"], "flujo_veh_h": flujo.round(1).to_numpy(),
        "exposicion_peatonal": peat.round(3).to_numpy(),
        "deterioro_infra": np.clip(r.beta(2, 5, len(seg)) + 0.1, 0.02, 0.95).round(3),   # URB-01 no lo trae: simulado
        "nodo_origen": seg["id_nodo_origen"], "nodo_destino": seg["id_nodo_destino"],
        "lat": lat_c, "lon": lon_c,
        "lat_o": ends.map(lambda e: e[0][1]), "lon_o": ends.map(lambda e: e[0][0]),
        "lat_d": ends.map(lambda e: e[1][1]), "lon_d": ends.map(lambda e: e[1][0])})
    df["indice_criticidad_fci03"] = sim.indice_estatico(sim.construir_red(df)).round(2)
    f = pd.to_datetime(afo["fecha_hora"])
    hist = pd.DataFrame({"id_segmento": afo["id_segmento"], "dia": (f.dt.normalize() - f.dt.normalize().min()).dt.days,
                         "hora": f.dt.hour, "flujo_veh_h": afo["vehiculos_hora"].astype(float)})
    return df, hist

# ---------------- Carga del insumo ----------------
historico = None
if ORIGEN_INSUMO == "csv":
    insumo = pd.read_csv(RUTA_CSV)
elif ORIGEN_INSUMO == "urb01":
    insumo, historico = cargar_desde_urb01(RUTA_URB01)
else:
    insumo = generar_insumo_sintetico(SEMILLA)

problemas = sim.validar_insumo(insumo)
print("Validación del insumo:", "OK, sin problemas" if not problemas else problemas)
red = sim.construir_red(insumo)
if historico is None:
    historico = generar_historico_sintetico(red)
insumo.to_csv(SALIDA / "insumo_fci_segmentos.csv", index=False)

print(f"Origen: {ORIGEN_INSUMO} | tramos: {red.n} | zonas: {sorted(set(red.zona))} | "
      f"vecinos por tramo (media): {np.mean([len(v) for v in red.vecinos]):.1f}")
display(insumo.drop(columns=["lat_o", "lon_o", "lat_d", "lon_d"], errors="ignore").head())

### 4.3 Qué pasa si el insumo viene mal

Parte de la HU-01 es **rechazar con claridad** un insumo defectuoso en lugar de simular con datos corruptos. Se comprueba con tres insumos dañados a propósito:

In [ ]:
casos = {
    "Falta una columna": insumo.drop(columns=["flujo_veh_h"]),
    "Exposición peatonal fuera de [0, 1]": insumo.assign(exposicion_peatonal=insumo["exposicion_peatonal"] * 3),
    "Identificadores repetidos": pd.concat([insumo, insumo.head(2)]),
}
for nombre, df_malo in casos.items():
    print(f"- {nombre}: {sim.validar_insumo(df_malo)}")

### 4.4 Escenario base en el mapa

Criticidad estática de cada tramo (el índice base del insumo). Los tramos más rojos son los candidatos naturales para intervenir.

In [ ]:
import folium, branca

CMAP_CRIT = branca.colormap.LinearColormap(["#1a9850", "#fee08b", "#d73027"],
                                           vmin=float(insumo["indice_criticidad_fci03"].quantile(0.02)),
                                           vmax=float(insumo["indice_criticidad_fci03"].quantile(0.98)),
                                           caption="Criticidad (0-100)")
CMAP_DELTA = branca.colormap.LinearColormap(["#1a9850", "#f7f7f7", "#d73027"], vmin=-6, vmax=6,
                                            caption="Cambio de criticidad vs. base (verde = mejora)")

def crear_mapa():
    return folium.Map(location=[insumo["lat"].mean(), insumo["lon"].mean()], zoom_start=16,
                      tiles="cartodbpositron")

def capa_tramos(m, nombre, valores, cmap, objetivo=(), mostrar=True, etiqueta="valor"):
    # Dibuja cada tramo como línea (o punto, si el insumo no trae extremos) coloreada por 'valores'
    fg = folium.FeatureGroup(name=nombre, show=mostrar)
    objetivo = set(objetivo)
    for i, r in enumerate(insumo.itertuples()):
        tip = f"{r.id_segmento} · {r.tipo_via} · zona {r.id_zona} · {etiqueta}: {valores[i]:.1f}"
        col, grosor = cmap(valores[i]), (7 if r.id_segmento in objetivo else 3)
        if hasattr(r, "lat_o"):
            folium.PolyLine([(r.lat_o, r.lon_o), (r.lat_d, r.lon_d)], color=col, weight=grosor,
                            opacity=0.9, tooltip=tip).add_to(fg)
        else:
            folium.CircleMarker((r.lat, r.lon), radius=grosor, color=col, fill=True, tooltip=tip).add_to(fg)
    fg.add_to(m)
    return fg

m0 = crear_mapa()
capa_tramos(m0, "Criticidad base", insumo["indice_criticidad_fci03"].to_numpy(), CMAP_CRIT, etiqueta="criticidad")
CMAP_CRIT.add_to(m0)
m0

## 5. Verificación del modelo (pruebas modulares)

**Verificar** es comprobar que el código implementa correctamente el diseño (distinto de *validar*, que compara contra la realidad, §6). El anteproyecto incluye *pruebas de regresión y modulares* entre sus prácticas; estas son las pruebas base del módulo:

| Prueba | Qué garantiza |
|---|---|
| T1 Determinismo | Misma semilla, mismo resultado (reproducibilidad científica) |
| T2 Intervención nula | Una intervención con efecto 0 reproduce exactamente el escenario base |
| T3 Conservación del flujo | Desviar o restringir **no crea ni destruye** vehículos: solo los mueve |
| T4 Límites del índice | La criticidad siempre está entre 0 y 100 |
| T5 Monotonía | Más desvío implica menos flujo en el tramo intervenido |
| T6 Insumo inválido | Un insumo dañado se rechaza con error |
| T7 Números aleatorios comunes | Fuera de la zona de influencia, base e intervenido ven la misma demanda |

In [ ]:
obj = sim.seleccionar_objetivo(red, k=K_OBJETIVO)
cfg_base = sim.ConfigSim()
base_sc = sim.Escenario("Base")
resultados_test = []

def prueba(nombre, condicion, detalle=""):
    resultados_test.append({"prueba": nombre, "resultado": "PASA" if condicion else "FALLA", "detalle": detalle})

# T1
a = sim.simular(red, sim.Escenario("X", [sim.Intervencion("zona_peatonal", obj)]), 11, cfg_base)
b = sim.simular(red, sim.Escenario("X", [sim.Intervencion("zona_peatonal", obj)]), 11, cfg_base)
prueba("T1 Determinismo", np.array_equal(a.crit_t, b.crit_t))

# T2
nula = sim.Escenario("Nula", [sim.Intervencion("desvio_ruta", obj, {"fraccion_desvio": 0.0})])
prueba("T2 Intervención nula = base",
       np.allclose(sim.simular(red, nula, 3, cfg_base).crit_t, sim.simular(red, base_sc, 3, cfg_base).crit_t))

# T3
ok3, detalle3 = True, []
for tipo in ("desvio_ruta", "restriccion_carril", "zona_peatonal"):
    est = {"flujo": red.flujo.copy(), "cap": red.cap.copy(), "peat": red.peat.copy(), "infra": red.infra.copy()}
    sim._aplicar(est, sim.Intervencion(tipo, obj), red)
    dif = abs(est["flujo"].sum() - red.flujo.sum())
    ok3 &= dif < 1e-6
    detalle3.append(f"{tipo}: Δ={dif:.1e}")
prueba("T3 Conservación del flujo", ok3, "; ".join(detalle3))

# T4
c = sim.simular(red, sim.Escenario("X", [sim.Intervencion("restriccion_carril", obj)]), 5, cfg_base).crit_t
prueba("T4 Criticidad en [0, 100]", (c.min() >= 0) and (c.max() <= 100), f"min={c.min():.1f}, max={c.max():.1f}")

# T5
def flujo_obj(frac):
    e = {"flujo": red.flujo.copy(), "cap": red.cap.copy(), "peat": red.peat.copy(), "infra": red.infra.copy()}
    sim._aplicar(e, sim.Intervencion("desvio_ruta", obj, {"fraccion_desvio": frac}), red)
    return e["flujo"][red.indices(obj)].sum()
prueba("T5 Monotonía del desvío", flujo_obj(0.2) > flujo_obj(0.5) > flujo_obj(0.8))

# T6
try:
    sim.construir_red(insumo.drop(columns=["flujo_veh_h"]))
    prueba("T6 Insumo inválido rechazado", False)
except ValueError:
    prueba("T6 Insumo inválido rechazado", True)

# T7
sc = sim.Escenario("S", [sim.Intervencion("senalizacion", obj)])
rb, rs = sim.simular(red, base_sc, 9, cfg_base), sim.simular(red, sc, 9, cfg_base)
_, zona_infl = sim.zonas_escenario(red, sc)
lejos = np.setdiff1d(np.arange(red.n), zona_infl)
prueba("T7 Mismo ruido de demanda", len(lejos) > 0 and np.allclose(rb.flujo_t[:, lejos], rs.flujo_t[:, lejos], rtol=0.25),
       f"{len(lejos)} tramos fuera de la zona de influencia")

tests = pd.DataFrame(resultados_test)
display(tests)
assert (tests["resultado"] == "PASA").all(), "Hay pruebas de verificación que fallan"
print("Verificación completa: todas las pruebas pasan.")

## 6. Validación del escenario base (K-S y calibración)

Antes de comparar intervenciones hay que asegurar que **el escenario base se parece a lo observado**. El anteproyecto lo plantea con una prueba de bondad de ajuste de **Kolmogorov-Smirnov (K-S)** frente a datos históricos. Se hace en tres pasos, separando datos de calibración y de validación para no "trampear":

1. **Calibrar** la variabilidad de la demanda (`sigma_demanda`) con los **primeros 60 % de los días** del histórico.
2. **Validar con K-S** en los días restantes: se compara la distribución del flujo horario simulado contra la observada. Hipótesis nula: ambas provienen de la misma distribución. Si `p > 0.05` **no se rechaza** el ajuste.
3. **Contrastar con FCI-03:** la correlación de Spearman entre el índice base de FCI-03 y la criticidad media simulada debe ser alta (≥ 0.90), señal de que el simulador reproduce el ranking de criticidad de los tramos.

> En el piloto el "histórico" es sintético (generado con una variabilidad real distinta a la supuesta por defecto, para que la calibración tenga sentido). En el proyecto se reemplaza por los aforos de ATM / FCI-01.
>
> **Si el K-S se rechaza, es información útil, no un error.** Por ejemplo, con `ORIGEN_INSUMO = "urb01"` el histórico son los aforos de URB-01, que cubren solo 8 horas diurnas con un perfil distinto al del simulador; el K-S lo detecta. La respuesta correcta es recalibrar el perfil horario (`PERFIL_VEH`) y la capacidad con los datos reales antes de interpretar los escenarios.

In [ ]:
# --- 1) Calibración de sigma con los primeros días ---
dias = np.sort(historico["dia"].unique())
corte_dia = dias[int(np.ceil(0.6 * len(dias))) - 1]
h_cal = historico[historico["dia"] <= corte_dia].copy()
h_val = historico[historico["dia"] > corte_dia].copy()

pos = {k: i for i, k in enumerate(red.ids)}
def esperado(h):
    return red.flujo[h["id_segmento"].map(pos).to_numpy()] * sim.PERFIL_VEH[h["hora"].to_numpy()]

ratio = np.log(h_cal["flujo_veh_h"].to_numpy() / esperado(h_cal))
SIGMA_CAL = float(np.clip(ratio.std(), 0.05, 1.0))
print(f"Calibración con {len(h_cal):,} observaciones (días ≤ {corte_dia}): sigma_demanda = {SIGMA_CAL:.3f} "
      f"(supuesto inicial: {cfg_base.sigma_demanda}) | sesgo log medio = {ratio.mean() + 0.5 * SIGMA_CAL**2:+.3f}")

# --- 2) K-S en los días de validación ---
cfg_cal = sim.ConfigSim(sigma_demanda=SIGMA_CAL)
idx_v = h_val["id_segmento"].map(pos).to_numpy()
hor_v = h_val["hora"].to_numpy()
sim_muestra = np.concatenate([sim.simular(red, base_sc, 100 + s, cfg_cal).flujo_t[hor_v, idx_v] for s in range(20)])
ks = stats.ks_2samp(h_val["flujo_veh_h"].to_numpy(), sim_muestra)

# --- 3) Correlación con el índice de FCI-03 ---
crit_sim = np.mean([sim.simular(red, base_sc, 200 + s, cfg_cal).crit_t.mean(0) for s in range(30)], axis=0)
rho = stats.spearmanr(red.indice_fci, crit_sim)[0]

validacion = pd.DataFrame([
    ("K-S: estadístico D (observado vs. simulado)", round(ks.statistic, 4), "—", ""),
    ("K-S: valor p", round(ks.pvalue, 4), "> 0.05", "CUMPLE" if ks.pvalue > 0.05 else "NO CUMPLE"),
    ("Correlación de Spearman con FCI-03", round(rho, 3), "≥ 0.90", "CUMPLE" if rho >= 0.90 else "NO CUMPLE"),
], columns=["prueba", "valor", "criterio", "veredicto"])
display(validacion)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(h_val["flujo_veh_h"], bins=40, density=True, alpha=0.55, label="Observado (validación)")
ax[0].hist(sim_muestra, bins=40, density=True, alpha=0.55, label="Simulado (base)")
ax[0].set_title(f"Distribución del flujo horario · K-S p = {ks.pvalue:.3f}"); ax[0].set_xlabel("veh/h"); ax[0].legend()
ax[1].scatter(red.indice_fci, crit_sim, s=12, alpha=0.7)
ax[1].set_xlabel("Índice base FCI-03"); ax[1].set_ylabel("Criticidad media simulada")
ax[1].set_title(f"Concordancia de ranking · ρ = {rho:.2f}")
plt.tight_layout(); plt.show()
if ks.pvalue <= 0.05:
    print("AVISO: el ajuste K-S se rechaza. Revisar el perfil horario, la capacidad o recalibrar antes de interpretar los escenarios.")

## 7. Catálogo de escenarios (HU-02)

Un **escenario** es un conjunto de intervenciones, cada una con: tipo, tramos objetivo, parámetros y ventana horaria. Se definen como **datos**, no como código, de modo que un planificador urbano (o el dashboard) pueda crearlos y el módulo solo los ejecute.

**Selección de tramos objetivo:** regla simple y transparente: los `K_OBJETIVO` tramos de **mayor criticidad base**. Es la forma más natural de priorizar una intervención preventiva, y deja abierta la pregunta de si ese esfuerzo reduce el riesgo en la zona o solo lo desplaza.

| Escenario | Intervención | Ventana |
|---|---|---|
| **Desvío de ruta** | 50 % del flujo de los tramos críticos pasa a vías vecinas | todo el día |
| **Restricción de carril** | −30 % de capacidad, con demanda elástica | 06:00–20:00 |
| **Zona peatonal** | −80 % de tráfico y −60 % de exposición peatonal | 06:00–22:00 |
| **Señalización** | −35 % de deterioro / mejora de infraestructura | todo el día |
| **Combinado** | Zona peatonal en los 6 tramos más críticos + señalización en los 12 | combinado |

In [ ]:
ESCENARIOS = [
    sim.Escenario("Desvío de ruta", [sim.Intervencion("desvio_ruta", obj)],
                  "Desvía el 50 % del flujo de los tramos más críticos hacia vías vecinas."),
    sim.Escenario("Restricción de carril", [sim.Intervencion("restriccion_carril", obj, inicio_h=6, fin_h=20)],
                  "Reduce la capacidad 30 % de 06:00 a 20:00; la demanda se ajusta."),
    sim.Escenario("Zona peatonal", [sim.Intervencion("zona_peatonal", obj, inicio_h=6, fin_h=22)],
                  "Restringe el tráfico vehicular y prioriza peatones de 06:00 a 22:00."),
    sim.Escenario("Señalización", [sim.Intervencion("senalizacion", obj)],
                  "Mejora la señalización e infraestructura en los tramos críticos."),
    sim.Escenario("Combinado", [sim.Intervencion("zona_peatonal", obj[:6], inicio_h=6, fin_h=22),
                                sim.Intervencion("senalizacion", obj)],
                  "Zona peatonal en los 6 tramos más críticos más señalización en los 12."),
]

cat = pd.DataFrame([{"escenario": e.nombre, "intervencion": iv.tipo, "n_tramos": len(iv.objetivo),
                     "ventana": f"{int(iv.inicio_h):02d}:00-{int(iv.fin_h) % 24:02d}:00" if iv.fin_h < 24 else "todo el día",
                     "parametros": iv.params} for e in ESCENARIOS for iv in e.intervenciones])
cat.to_csv(SALIDA / "catalogo_escenarios.csv", index=False, encoding="utf-8-sig")
display(cat)
print("Tramos objetivo (por criticidad base):", obj)

## 8. Una corrida DES paso a paso

Antes de promediar cientos de réplicas conviene **ver una sola**. Se simula un día con la intervención *Zona peatonal* y con la misma semilla para el escenario base (números aleatorios comunes). La bitácora de eventos muestra cómo la cola procesa, en orden de tiempo, el inicio y fin de la intervención y los incidentes.

Los gráficos separan el efecto en tres grupos de tramos, que es el corazón del análisis preventivo:

- **Tramos intervenidos:** donde se aplica la medida (debería mejorar).
- **Vecinos directos:** reciben el flujo desviado (podrían empeorar: *el conflicto se traslada*).
- **Resto de la zona:** referencia sin cambio directo.

In [ ]:
esc_demo = ESCENARIOS[2]            # Zona peatonal
res_b = sim.simular(red, base_sc, semilla=1, cfg=cfg_cal, registrar=True)
res_e = sim.simular(red, esc_demo, semilla=1, cfg=cfg_cal, registrar=True)

print(f"Escenario '{esc_demo.nombre}' · semilla 1 · incidentes en el día: {res_e.n_incidentes}")
display(pd.DataFrame(res_e.log).head(12))

obj_i, infl_i = sim.zonas_escenario(red, esc_demo)
vec_i = np.setdiff1d(infl_i, obj_i)
resto_i = np.setdiff1d(np.arange(red.n), infl_i)
t_inc = [e["t_h"] for e in res_e.log if e["evento"] == "INCIDENTE"]

fig, axs = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (nombre, idx) in zip(axs, [("Tramos intervenidos", obj_i), ("Vecinos directos", vec_i), ("Resto de la zona", resto_i)]):
    ax.plot(res_b.crit_t[:, idx].mean(1), lw=2, label="Base")
    ax.plot(res_e.crit_t[:, idx].mean(1), lw=2, label=esc_demo.nombre)
    for t in t_inc:
        ax.axvline(t, color="gray", alpha=0.25)
    ax.set_title(f"{nombre} (n={len(idx)})"); ax.set_xlabel("hora del día")
axs[0].set_ylabel("Criticidad media"); axs[0].legend()
plt.suptitle("Una réplica: criticidad por hora (líneas grises = llegada de incidentes)", y=1.02)
plt.tight_layout(); plt.show()

## 9. Monte Carlo y comparación de escenarios (HU-03)

Se ejecutan **`N_REP` réplicas** de cada escenario. En cada réplica varían, de forma aleatoria pero **idéntica para todos los escenarios**:

- la demanda de cada tramo y hora (variabilidad calibrada en §6),
- la llegada, ubicación y duración de los incidentes,
- la **efectividad de cada intervención** (`incertidumbre_parametros = ±20 %`): nadie sabe exactamente cuánto desviará un desvío o cuánto mejorará una señalización.

### Indicadores (matriz de operacionalización del anteproyecto)

| Columna | Significado |
|---|---|
| `reduccion_obj_pct` | % de reducción de criticidad **en los tramos intervenidos** |
| `reduccion_infl_pct` | % de reducción **neta en la zona de influencia** (intervenidos + vecinos). Es el indicador principal: descuenta lo que se traslada a los vecinos |
| `reduccion_ciudad_pct` | % de reducción en toda el área piloto |
| `ic95_inf`, `ic95_sup` | Intervalo de confianza 95 % de `reduccion_infl_pct` |
| `prob_mejora` | Proporción de réplicas en que el escenario reduce la criticidad de la zona de influencia |
| `desplazamiento_max` | Mayor aumento de criticidad (puntos) en algún tramo no intervenido |
| `segmentos_empeorados` | Tramos que empeoran más de 1 punto |

In [ ]:
cfg_mc = sim.ConfigSim(sigma_demanda=SIGMA_CAL, incertidumbre_parametros=0.20)
t0 = time.time()
mc = sim.monte_carlo(red, ESCENARIOS, n_rep=N_REP, cfg=cfg_mc, semilla0=1000)
print(f"{len(ESCENARIOS) + 1} escenarios x {N_REP} réplicas en {time.time() - t0:.1f} s")

tabla = mc.tabla()
def lectura(f):
    if f.name == "Base":                       return "referencia"
    if f["reduccion_infl_pct"] <= 0:           return "no reduce la criticidad de la zona"
    if f["segmentos_empeorados"] >= 3:         return "reduce donde se aplica, pero empeora tramos vecinos (traslado)"
    return "reduce sin efectos adversos relevantes en vecinos"
tabla["lectura"] = tabla.apply(lectura, axis=1)
display(tabla.round(2))

In [ ]:
t = tabla.drop(index="Base")
x = np.arange(len(t)); w = 0.27
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(x - w, t["reduccion_obj_pct"], w, label="Tramos intervenidos", color="#1b9e77")
ax.bar(x, t["reduccion_infl_pct"], w, label="Zona de influencia (neto)", color="#d95f02",
       yerr=[t["reduccion_infl_pct"] - t["ic95_inf"], t["ic95_sup"] - t["reduccion_infl_pct"]], capsize=3)
ax.bar(x + w, t["reduccion_ciudad_pct"], w, label="Toda el área piloto", color="#7570b3")
ax.axhline(0, color="k", lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(t.index, rotation=15); ax.set_ylabel("Reducción de criticidad (%)")
ax.set_title("Reducción estimada de criticidad según el alcance de la medición"); ax.legend()
plt.tight_layout(); plt.show()

### Cómo leer la tabla y el gráfico

- **Las tres barras cuentan historias distintas.** Una medida puede mejorar mucho los tramos intervenidos y casi nada la zona (se trasladó el conflicto), o incluso empeorarla. Esa diferencia justifica el diseño del módulo: sin simulación previa, la política *parecería* exitosa mirando solo los tramos tratados.
- **Los intervalos de confianza son muy estrechos.** No es un error: por el uso de números aleatorios comunes la comparación es pareada y el ruido compartido se cancela. Significa que, **bajo los supuestos del modelo**, el orden de los escenarios es estable. No significa que los supuestos sean ciertos; eso lo prueban la validación (§6) y la sensibilidad (§11).
- **Los porcentajes dependen de supuestos de efectividad** (tabla de §3.2) que aún no se calibran con evidencia de campo.

## 10. Diseño de experimentos: ANOVA / Kruskal-Wallis

El anteproyecto compromete un **Diseño de Experimentos (DOE)** para certificar que las diferencias entre políticas tienen fundamento estadístico. Aquí el **factor** es el escenario (5 niveles) y la **respuesta** es `reduccion_infl_pct` de cada réplica.

| Prueba | Para qué |
|---|---|
| **Shapiro-Wilk** (por escenario) y **Levene** | Comprobar los supuestos de normalidad y varianzas iguales del ANOVA |
| **ANOVA de un factor** | ¿Difieren las medias de los escenarios? |
| **Kruskal-Wallis** | Alternativa sin supuesto de normalidad (la que pide el anteproyecto si los supuestos fallan) |
| **Friedman** | Como las réplicas son **pareadas** (misma semilla en todos los escenarios), esta es la prueba que respeta el diseño |
| **η² (eta cuadrado)** | Tamaño del efecto: qué proporción de la variabilidad explica el escenario |
| **Tukey HSD** | ¿Qué pares de escenarios difieren? |

In [ ]:
piv = mc.resumen[mc.resumen["escenario"] != "Base"].pivot(index="rep", columns="escenario", values="reduccion_infl_pct")
grupos = [piv[c].to_numpy() for c in piv.columns]

normalidad = {c: stats.shapiro(piv[c]).pvalue for c in piv.columns}
lev = stats.levene(*grupos)
f_an = stats.f_oneway(*grupos)
kw = stats.kruskal(*grupos)
fr = stats.friedmanchisquare(*grupos)
gm = np.mean(np.concatenate(grupos))
eta2 = sum(len(g) * (g.mean() - gm) ** 2 for g in grupos) / sum(((g - gm) ** 2).sum() for g in grupos)

doe = pd.DataFrame([
    ("Shapiro-Wilk (mínimo p entre escenarios)", min(normalidad.values()), "p > 0.05 → normal"),
    ("Levene (homogeneidad de varianzas)", lev.pvalue, "p > 0.05 → varianzas iguales"),
    ("ANOVA de un factor", f_an.pvalue, "p < 0.05 → medias distintas"),
    ("Kruskal-Wallis", kw.pvalue, "p < 0.05 → distribuciones distintas"),
    ("Friedman (diseño pareado)", fr.pvalue, "p < 0.05 → escenarios distintos"),
], columns=["prueba", "valor_p", "interpretación"])
doe["valor_p"] = doe["valor_p"].map(lambda v: f"{v:.2e}")
display(doe)
print(f"η² = {eta2:.3f}  → el escenario explica {eta2:.1%} de la variabilidad de la reducción de criticidad")
if min(normalidad.values()) < 0.05 or lev.pvalue < 0.05:
    print("Los supuestos del ANOVA no se cumplen de forma estricta: se prioriza Kruskal-Wallis / Friedman.")

tuk = stats.tukey_hsd(*grupos)
display(pd.DataFrame(tuk.pvalue, index=piv.columns, columns=piv.columns).round(4))

> **Significancia estadística no es relevancia práctica.** Con cientos de réplicas casi cualquier diferencia sale "significativa". Por eso se reporta también **η²** y, sobre todo, la **magnitud** de la reducción (§9). La matriz de Tukey indica qué pares de escenarios son distinguibles (valores p cercanos a 0); un valor alto indica que esos dos escenarios son estadísticamente equivalentes.

## 11. Análisis de sensibilidad y robustez del ranking

El anteproyecto exige medir **qué tan robustos son los hallazgos ante pequeñas fluctuaciones en los parámetros**. Se hacen tres análisis sobre el escenario *Zona peatonal*; la métrica de salida es la reducción de criticidad en la zona de influencia:

1. **Un parámetro a la vez (OAT, diagrama de tornado):** cada parámetro se mueve entre su mínimo y máximo, dejando los demás en su valor central. Muestra *cuánto* mueve el resultado cada supuesto.
2. **Sensibilidad global (hipercubo latino + Spearman):** todos los parámetros varían **a la vez**. Detecta efectos combinados y su dirección (signo).
3. **Estabilidad del ranking:** ¿cambia el orden de los escenarios si la incertidumbre sobre la efectividad de las medidas pasa de 0 % a 20 % y 40 %? Se mide con la **τ de Kendall** (1 = mismo orden).

In [ ]:
esc_sens = ESCENARIOS[2]                      # Zona peatonal -> intervención 0
rangos = {
    "iv0.restriccion_vehicular": (0.50, 0.95),    # efectividad de la medida
    "iv0.reduccion_exposicion":  (0.30, 0.80),
    "cfg.sigma_demanda":         (0.10, 0.40),    # variabilidad de la demanda
    "cfg.tasa_incidentes_dia":   (2.0, 12.0),     # incidentes por día
    "cfg.derrame_incidente":     (0.05, 0.30),
    "cfg.factor_cap_incidente":  (0.30, 0.80),
}
t0 = time.time()
oat = sim.sensibilidad_oat(red, esc_sens, cfg_cal, rangos, n_rep=N_REP_SENS)
glob = sim.sensibilidad_global(red, esc_sens, cfg_cal, rangos, n_muestras=64, n_rep=8)
print(f"Sensibilidad calculada en {time.time() - t0:.1f} s")
display(oat.round(3))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
o = oat.iloc[::-1]
c0 = o["reduccion_central"].iloc[0]
ax[0].barh(o["parametro"], o["reduccion_en_max"] - c0, left=c0, color="#d95f02", alpha=0.8, label="valor máximo")
ax[0].barh(o["parametro"], o["reduccion_en_min"] - c0, left=c0, color="#1b9e77", alpha=0.8, label="valor mínimo")
ax[0].axvline(c0, color="k", lw=1); ax[0].set_xlabel("Reducción en zona de influencia (%)")
ax[0].set_title("Tornado (un parámetro a la vez)"); ax[0].legend()
g = glob.iloc[::-1]
ax[1].barh(g["parametro"], g["spearman"], color=np.where(g["spearman"] >= 0, "#3b6ea5", "#b2182b"))
ax[1].axvline(0, color="k", lw=1); ax[1].set_xlabel("Correlación de Spearman con la reducción")
ax[1].set_title("Sensibilidad global (hipercubo latino)")
plt.tight_layout(); plt.show()

In [ ]:
# --- Estabilidad del ranking ante la incertidumbre sobre la efectividad ---
ranks = {}
for inc in (0.0, 0.2, 0.4):
    c = sim.ConfigSim(sigma_demanda=SIGMA_CAL, incertidumbre_parametros=inc)
    t = sim.monte_carlo(red, ESCENARIOS, n_rep=60, cfg=c, semilla0=3000).tabla().drop(index="Base")
    ranks[f"±{int(inc * 100)} %"] = t["reduccion_infl_pct"].rank(ascending=False).astype(int)
rk = pd.DataFrame(ranks)
display(rk.sort_values(rk.columns[0]))
tau = {f"{a} vs {b}": stats.kendalltau(rk[a], rk[b])[0] for a, b in [(rk.columns[0], rk.columns[1]), (rk.columns[0], rk.columns[2])]}
print("τ de Kendall entre rankings:", {k: round(v, 2) for k, v in tau.items()})

### Cómo interpretar

- En el **tornado**, los parámetros con barras más largas son los supuestos que más condicionan el resultado: ahí conviene invertir esfuerzo de **calibración con datos reales** primero. Los de barra corta pueden dejarse en su valor por defecto sin riesgo.
- En la **sensibilidad global**, el signo importa: un valor positivo indica que subir el parámetro aumenta la reducción estimada.
- Si la **τ de Kendall** es cercana a 1, el **ranking de escenarios es robusto**: la recomendación (qué alternativa es mejor) no depende de la incertidumbre sobre la efectividad. Si cae, la recomendación es frágil y debe presentarse con cautela.

## 12. Tablero y mapas (HU-04)

La HU-04 pide una **comparativa gráfica base vs. intervenido** para identificar rápidamente la mejor alternativa. Se entregan dos productos interactivos:

1. **Tablero Plotly** (`tablero_urb05.html`): ranking de escenarios con intervalos de confianza, perfil horario de criticidad y mapa de calor por zona. Se puede pasar el cursor para ver valores y hacer clic en la leyenda para ocultar series.
2. **Mapa Folium** (`mapa_escenarios.html`): una capa por escenario con el **cambio de criticidad por tramo** (verde = mejora, rojo = empeora). Se alternan con el control de capas. Los **tramos intervenidos se dibujan más gruesos**: de un vistazo se ve si el rojo aparece alrededor, es decir, si el conflicto se trasladó.

> El anteproyecto menciona también exportar a **Power BI Desktop**; las tablas de §13 están en formato largo precisamente para eso.

In [ ]:
import plotly.graph_objects as go

t = tabla.drop(index="Base")
fig1 = go.Figure()
fig1.add_bar(name="Tramos intervenidos", x=t.index, y=t["reduccion_obj_pct"], marker_color="#1b9e77")
fig1.add_bar(name="Zona de influencia (neto)", x=t.index, y=t["reduccion_infl_pct"], marker_color="#d95f02",
             error_y=dict(type="data", symmetric=False, array=t["ic95_sup"] - t["reduccion_infl_pct"],
                          arrayminus=t["reduccion_infl_pct"] - t["ic95_inf"]))
fig1.add_bar(name="Toda el área piloto", x=t.index, y=t["reduccion_ciudad_pct"], marker_color="#7570b3")
fig1.update_layout(barmode="group", title="Ranking de escenarios: reducción estimada de criticidad (%)",
                   yaxis_title="Reducción (%)", legend_orientation="h", legend_y=-0.25)

fig2 = go.Figure()
for nombre, serie in mc.crit_hora.items():
    fig2.add_scatter(x=list(range(24)), y=serie, mode="lines", name=nombre,
                     line=dict(width=4 if nombre == "Base" else 2, dash="dot" if nombre == "Base" else "solid"))
fig2.update_layout(title="Criticidad media por hora del día (promedio de réplicas)",
                   xaxis_title="Hora", yaxis_title="Criticidad media (0-100)")

zonas = sorted(set(red.zona))
base_seg = mc.crit_seg["Base"]
nombres_esc = [n for n in mc.crit_seg if n != "Base"]
Z = [[float((mc.crit_seg[n] - base_seg)[red.zona == z].mean()) for n in nombres_esc] for z in zonas]
fig3 = go.Figure(go.Heatmap(z=Z, x=nombres_esc, y=zonas, colorscale="RdYlGn_r", zmid=0,
                            colorbar_title="Δ criticidad", text=[[f"{v:+.2f}" for v in r] for r in Z],
                            texttemplate="%{text}"))
fig3.update_layout(title="Cambio medio de criticidad por zona (verde = mejora, rojo = empeora)")

for f in (fig1, fig2, fig3):
    f.show()

partes = [f.to_html(full_html=False, include_plotlyjs=("cdn" if i == 0 else False)) for i, f in enumerate([fig1, fig2, fig3])]
html = ("<html><head><meta charset='utf-8'><title>Tablero URB-05</title></head>"
        "<body style='font-family:sans-serif;max-width:1000px;margin:auto'>"
        "<h1>URB-05 · Simulación de escenarios preventivos</h1>"
        "<p>Estimaciones bajo supuestos documentados con datos simulados. No son predicciones.</p>" + "".join(partes) + "</body></html>")
(SALIDA / "tablero_urb05.html").write_text(html, encoding="utf-8")
print("Guardado:", SALIDA / "tablero_urb05.html")

In [ ]:
m = crear_mapa()
capa_tramos(m, "Criticidad base", mc.crit_seg["Base"], CMAP_CRIT, mostrar=False, etiqueta="criticidad")
for k, e in enumerate(ESCENARIOS):
    delta = mc.crit_seg[e.nombre] - mc.crit_seg["Base"]
    capa_tramos(m, f"Δ {e.nombre}", delta, CMAP_DELTA, objetivo=mc.objetivos[e.nombre], mostrar=(k == 2), etiqueta="Δ criticidad")
CMAP_DELTA.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m.save(SALIDA / "mapa_escenarios.html")
m

## 13. Exportación e integración con el dashboard / API (HU-05)

El anteproyecto pide exportar resultados **"mediante formatos estandarizados (API)"** para integrarlos al gemelo digital global. Se entregan tres niveles:

| Archivo | Formato | Uso |
|---|---|---|
| `resultados_urb05.json` | JSON con esquema versionado (`risktwin.urb05.resultados/1.0`) | **Contrato con la API / dashboard de FCI** |
| `resultados_escenarios.csv`, `resultados_replicas.csv` | CSV | Análisis estadístico, anexos de tesis |
| `resultados_por_tramo.csv` + `dim_tramos.csv` | CSV en formato largo + tabla de dimensión | **Power BI** (relación por `id_segmento`) |
| `mapa_delta.geojson` | GeoJSON | Cualquier visor GIS / capa del dashboard |

**Importante para la integración:** el módulo expone `ejecutar_desde_dict(red, payload)`, que recibe un JSON de configuración (escenarios, réplicas, parámetros) y devuelve el JSON de resultados. Es el punto de entrada que una API consumiría sin conocer los detalles internos del simulador.

In [ ]:
# ---------- CSV ----------
tabla.reset_index().to_csv(SALIDA / "resultados_escenarios.csv", index=False, encoding="utf-8-sig")
mc.resumen.to_csv(SALIDA / "resultados_replicas.csv", index=False, encoding="utf-8-sig")
largo = pd.DataFrame([{"id_segmento": red.ids[i], "escenario": n, "criticidad_media": arr[i],
                       "delta_vs_base": arr[i] - mc.crit_seg["Base"][i],
                       "intervenido": red.ids[i] in set(mc.objetivos[n])}
                      for n, arr in mc.crit_seg.items() for i in range(red.n)])
largo.to_csv(SALIDA / "resultados_por_tramo.csv", index=False, encoding="utf-8-sig")
insumo[["id_segmento", "id_zona", "tipo_via", "longitud_m", "lat", "lon"]].to_csv(SALIDA / "dim_tramos.csv", index=False, encoding="utf-8-sig")

# ---------- JSON del contrato ----------
salida_json = sim.resultados_a_dict(red, mc, cfg_mc)
with open(SALIDA / "resultados_urb05.json", "w", encoding="utf-8") as f:
    json.dump(salida_json, f, ensure_ascii=False, indent=1)

# ---------- GeoJSON ----------
feats = []
for i, r in enumerate(insumo.itertuples()):
    geom = ({"type": "LineString", "coordinates": [[r.lon_o, r.lat_o], [r.lon_d, r.lat_d]]} if hasattr(r, "lat_o")
            else {"type": "Point", "coordinates": [r.lon, r.lat]})
    props = {"id_segmento": r.id_segmento, "id_zona": r.id_zona, "tipo_via": r.tipo_via,
             "crit_base": round(float(mc.crit_seg["Base"][i]), 3)}
    for n in nombres_esc:
        props["delta_" + n.replace(" ", "_")] = round(float(mc.crit_seg[n][i] - mc.crit_seg["Base"][i]), 3)
    feats.append({"type": "Feature", "geometry": geom, "properties": props})
with open(SALIDA / "mapa_delta.geojson", "w", encoding="utf-8") as f:
    json.dump({"type": "FeatureCollection", "features": feats}, f, ensure_ascii=False)

print("Exportado en", SALIDA)
print(sorted(p.name for p in SALIDA.iterdir() if p.is_file()))

### Contrato JSON y punto de entrada para la API

Se valida que la salida cumpla el contrato (claves obligatorias, tipos y valores finitos) y luego se simula la llamada que haría la API: un JSON de **entrada** con escenarios definidos como datos (los tramos objetivo pueden indicarse como lista de ids o como una **regla**, por ejemplo `{"k": 8, "zona": "Z1N"}`).

In [ ]:
def validar_contrato(d):
    err = []
    for k in ("esquema", "version_modulo", "generado", "configuracion", "escenarios", "resultados_segmento"):
        if k not in d:
            err.append(f"falta la clave '{k}'")
    if err:
        return err
    if d["esquema"] != sim.ESQUEMA_RESULTADOS:
        err.append("esquema distinto al esperado")
    req_e = {"escenario", "n_rep", "reduccion_obj_pct", "reduccion_infl_pct", "ic95_inf", "ic95_sup", "prob_mejora"}
    req_s = {"id_segmento", "id_zona", "escenario", "criticidad_media", "delta_vs_base"}
    for e in d["escenarios"]:
        if not req_e <= set(e):
            err.append(f"escenario incompleto: {e.get('escenario')}")
    for s in d["resultados_segmento"]:
        if not req_s <= set(s) or not np.isfinite(s["criticidad_media"]):
            err.append(f"registro de tramo inválido: {s.get('id_segmento')}")
            break
    return err

print("Contrato de la salida principal:", validar_contrato(salida_json) or "OK")

# ---- Simulación de una llamada de API ----
zona_ej = sorted(set(red.zona))[0]
payload = {
    "semilla": 1000, "n_rep": 30,
    "cfg": {"sigma_demanda": round(SIGMA_CAL, 3), "incertidumbre_parametros": 0.2},
    "escenarios": [
        {"nombre": "Señalización top-10", "descripcion": "Mejora de señalización en los 10 tramos más críticos",
         "intervenciones": [{"tipo": "senalizacion", "objetivo": {"k": 10}, "params": {"mejora_infra": 0.40}}]},
        {"nombre": f"Zona peatonal {zona_ej}",
         "intervenciones": [{"tipo": "zona_peatonal", "objetivo": {"k": 8, "zona": zona_ej}, "inicio_h": 6, "fin_h": 22}]},
    ],
}
respuesta = sim.ejecutar_desde_dict(red, payload)
print("Contrato de la respuesta de API:", validar_contrato(respuesta) or "OK")
display(pd.DataFrame(respuesta["escenarios"])[["escenario", "n_rep", "reduccion_obj_pct", "reduccion_infl_pct", "ic95_inf", "ic95_sup", "prob_mejora"]])

### Ejemplo de endpoint (no se ejecuta en Colab)

Así quedaría el módulo detrás de una API del proyecto, por ejemplo con FastAPI (`pip install fastapi uvicorn`):

```python
from fastapi import FastAPI
import pandas as pd, simulation as sim

app = FastAPI(title="URB-05 · Simulación de escenarios")
red = sim.construir_red(pd.read_csv("insumo_fci_segmentos.csv"))   # insumo entregado por FCI

@app.post("/simular")
def simular(payload: dict):
    return sim.ejecutar_desde_dict(red, payload)
```

**Power BI Desktop:** *Obtener datos → Texto/CSV* → cargar `resultados_por_tramo.csv`, `resultados_escenarios.csv` y `dim_tramos.csv`; crear la relación `id_segmento` entre la tabla de hechos y la dimensión; usar `lat`/`lon` en un visual de mapa coloreado por `delta_vs_base`, con una segmentación por `escenario`.

## 14. Pruebas de integración, regresión y estrés

La etapa final del roadmap del anteproyecto es *"integración y pruebas finales: pruebas de estrés y despliegue mediante API"*. Se comprueban tres cosas:

1. **Regresión / reproducibilidad:** la misma llamada de API devuelve **exactamente** los mismos resultados (se compara una huella SHA-256, ignorando la marca de tiempo).
2. **Integridad del contrato:** cantidad de registros esperada y coherencia lógica (el escenario base nunca difiere de sí mismo).
3. **Estrés y escalabilidad:** tiempo de ejecución con muchas réplicas y con una red **más de cinco veces mayor**.

In [ ]:
def huella(d):
    d = {k: v for k, v in d.items() if k != "generado"}
    return hashlib.sha256(json.dumps(d, sort_keys=True).encode()).hexdigest()[:16]

r1 = sim.ejecutar_desde_dict(red, payload)
r2 = sim.ejecutar_desde_dict(red, payload)
esperado_seg = red.n * (len(payload["escenarios"]) + 1)

# estrés: más réplicas
t0 = time.time(); sim.monte_carlo(red, ESCENARIOS, n_rep=500, cfg=cfg_mc); t_rep = time.time() - t0
# escalabilidad: red grande
grande = generar_insumo_sintetico(1, n=20)
red_g = sim.construir_red(grande)
obj_g = sim.seleccionar_objetivo(red_g, k=40)
esc_g = [sim.Escenario("Zona peatonal", [sim.Intervencion("zona_peatonal", obj_g)]),
         sim.Escenario("Señalización", [sim.Intervencion("senalizacion", obj_g)])]
t0 = time.time(); sim.monte_carlo(red_g, esc_g, n_rep=100, cfg=cfg_mc); t_grande = time.time() - t0

integracion = pd.DataFrame([
    ("Regresión: misma llamada, misma huella", huella(r1) == huella(r2), f"{huella(r1)}"),
    ("Contrato válido en ambas respuestas", not validar_contrato(r1) and not validar_contrato(r2), ""),
    ("Registros por tramo = tramos x (escenarios + base)", len(r1["resultados_segmento"]) == esperado_seg, f"{len(r1['resultados_segmento'])}"),
    ("El escenario base no difiere de sí mismo",
     all(s["delta_vs_base"] == 0 for s in r1["resultados_segmento"] if s["escenario"] == "Base"), ""),
    ("Estrés: 6 escenarios x 500 réplicas", t_rep < 120, f"{t_rep:.1f} s"),
    (f"Escalabilidad: red de {red_g.n} tramos, 3 escenarios x 100 réplicas", t_grande < 120, f"{t_grande:.1f} s"),
], columns=["prueba", "cumple", "detalle"])
display(integracion)
assert integracion["cumple"].all(), "Falló alguna prueba de integración"
print("Integración y regresión: todas las pruebas pasan.")

## 15. Conclusiones, limitaciones y próximos pasos

In [ ]:
paquete_src = SALIDA / "simulation.py"
shutil.copy("simulation.py", paquete_src)
zip_path = shutil.make_archive("urb05_paquete", "zip", SALIDA)
print("Paquete listo:", zip_path)
print(sorted(p.name for p in SALIDA.iterdir() if p.is_file()))
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("(Fuera de Colab: el archivo queda en el directorio de trabajo.)")

### Lista de verificación frente al anteproyecto

| Entregable / criterio | Estado en el piloto |
|---|---|
| Módulo de simulación basado en reglas (`simulation.py`) | ✔ §3 |
| Escenario base cargado y validado desde insumos de FCI (HU-01) | ✔ §4 y §6 |
| Escenarios preventivos con reglas y parámetros (HU-02, OE1) | ✔ §3 y §7 (4 tipos de intervención + combinado) |
| Ejecución DES y comparación contra el escenario base (OE2) | ✔ §8 y §9 |
| Monte Carlo para variabilidad (HU-03) | ✔ §9 |
| Validación K-S del escenario base | ✔ §6 |
| DOE con ANOVA / Kruskal-Wallis | ✔ §10 |
| Reducción estimada de criticidad y análisis de sensibilidad (OE3) | ✔ §9 y §11 |
| Tablero comparativo interactivo (HU-04) | ✔ §12 |
| Exportación estandarizada e integración API (HU-05, OE4) | ✔ §13 y §14 |
| Pruebas modulares, de regresión y de estrés | ✔ §5 y §14 |
| Documento de parámetros y supuestos | ✔ §3.2 (`supuestos_parametros.csv`) |

### Hallazgo metodológico que vale la pena destacar en la tesis

El piloto muestra el valor del módulo: **medir solo los tramos intervenidos da una imagen engañosa**. Una medida puede reducir mucho la criticidad donde se aplica y, al desviar el flujo, aumentarla en los vecinos. Al reportar *tramos intervenidos*, *zona de influencia* y *toda el área*, el módulo permite distinguir **reducir el riesgo** de **trasladarlo**, que es precisamente el problema que plantea la sección 1.2 del anteproyecto.

### Limitaciones que conviene declarar

1. **Insumos sintéticos.** Hasta recibir FCI-01/02/03, la red, los flujos y el histórico son generados. Los porcentajes de reducción **no son predicciones** del comportamiento de la ciudad.
2. **Efectividad de las intervenciones.** Los parámetros (§3.2) son supuestos del equipo. Se recomienda tomar rangos de la literatura (el anteproyecto cita trabajos de gemelos digitales y medidas de seguridad sustitutivas) o de intervenciones piloto, y reportarlos como rangos.
3. **Criticidad simplificada.** La fórmula de §3.1 es un modelo de trabajo; debe alinearse con el índice real de FCI-03 (pesos y componentes).
4. **Un día típico.** El horizonte es de 24 h laborables; no se modelan fines de semana, lluvia ni eventos especiales.
5. **Vecindad topológica simple.** El flujo desviado se reparte a los tramos que comparten nodo. Con el grafo de URB-02 se podrían usar rutas alternativas reales (caminos mínimos).
6. **Intervalos estrechos.** Por el diseño pareado, la precisión estadística es alta pero **no** garantiza validez externa.
7. **Medidas de seguridad sustitutivas.** El anteproyecto menciona indicadores como el tiempo mínimo de colisión; este piloto usa criticidad agregada, no trayectorias.

### Próximos pasos sugeridos

1. Recibir los insumos reales y ejecutar con `ORIGEN_INSUMO = "csv"` (mismo contrato). Si se desea probar la cadena completa con el módulo anterior, usar `"urb01"`.
2. Reemplazar el histórico sintético por **aforos de ATM** y repetir la calibración y el K-S de §6.
3. Definir con los planificadores urbanos **rangos realistas de efectividad** para cada intervención.
4. Usar el grafo de URB-02 para el reparto del flujo desviado y calcular el **impacto sistémico** de cada desvío.
5. Conectar `ejecutar_desde_dict` a la API del proyecto FCI y el tablero a Power BI.
6. Añadir fines de semana y condiciones de lluvia (variables de URB-01) como parámetros del escenario.

### Nota bibliográfica

El anteproyecto cita **Banks et al.** en las secciones 3.3 y 7.5, pero esa obra **no aparece en la lista de referencias** (sección 14). Conviene agregarla antes de la entrega, con la edición exacta que se haya consultado.